In [3]:
from typing import List, Dict, Any, Optional, Tuple
from app.vector_db import get_qa_chain

def _parse_lineage_page_content(page_content: str) -> Dict[str, Optional[str]]:
    """
    Parse a Document.page_content that was produced by _lineage_row_to_document/_row_to_document
    into a structured lineage dict. Unknown keys are ignored.
    """
    expected_keys = {
        "source_database",
        "source_schema",
        "source_table",
        "source_column",
        "target_database",
        "target_schema",
        "target_table",
        "target_column",
        "query_id",
        "query_type",
        "dbt_model_file_path",
        "dependency_score",
    }
    parsed: Dict[str, Optional[str]] = {k: None for k in expected_keys}
    for line in page_content.splitlines():
        if ": " not in line:
            continue
        key, val = line.split(": ", 1)
        key = key.strip()
        if key in parsed:
            parsed[key] = val.strip()
    return parsed

def extract_lineage(
    org_id: str,
    question: str,
    k: int = 8,
) -> Dict[str, Any]:
    """
    Answer a lineage question using the org's vector store and return:
      - answer: LLM synthesized answer
      - lineage_entries: parsed structured lineage rows from retrieved docs
      - sources: minimal source attribution for transparency
    """
    qa = get_qa_chain(org_id=org_id, k=k)
    # For modern LangChain, invoke with a dict to ensure sources are returned
    result = qa.invoke({"query": question})
    answer: str = result.get("result", "") or result.get("answer", "")

    source_docs = result.get("source_documents", []) or []
    lineage_entries: List[Dict[str, Optional[str]]] = []
    sources_min: List[Dict[str, Any]] = []

    for doc in source_docs:
        # Parse the page_content into structured lineage fields when possible
        parsed = _parse_lineage_page_content(getattr(doc, "page_content", "") or "")
        lineage_entries.append(parsed)

        # Minimal source metadata for attribution
        sources_min.append({
            "metadata": getattr(doc, "metadata", {}) or {},
            "preview": (getattr(doc, "page_content", "") or "").splitlines()[:6],
        })

    return {
        "answer": answer,
        "lineage_entries": lineage_entries,
        "sources": sources_min,
    }

E0000 00:00:1762424468.902905 7628451 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.
E0000 00:00:1762424468.906951 7628451 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


In [4]:
extract_lineage(org_id='76d33fb3-6062-456b-a211-4aec9971f8be',
question='can you give me the lineage for gl_summary table')

/Users/deepkatbamna/Desktop/query_guard/QueryGuard-Backend/app/vector_db.py:29: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  db = Chroma(


{'answer': 'Here is the lineage information for the `gl_summary` table:\n\n*   **source_database**: `transform_zone`\n*   **source_schema**: `edw`\n*   **source_table**: `gl_summary`\n*   **source_column**: `accountcd`\n    *   **target_database**: `transform_zone`\n    *   **target_schema**: `edw`\n    *   **target_table**: `allocationrule`\n    *   **target_column**: `allocationrulekey`\n*   **source_database**: `transform_zone`\n*   **source_schema**: `edw`\n*   **source_table**: `gl_summary`\n*   **source_column**: `accountcd`\n    *   **target_database**: `transform_zone`\n    *   **target_schema**: `edw`\n    *   **target_table**: `allocationrule`\n    *   **target_column**: `accountfrom`\n*   **source_database**: `transform_zone`\n*   **source_schema**: `edw`\n*   **source_table**: `gl_summary`\n*   **source_column**: `accountcd`\n    *   **target_database**: `transform_zone`\n    *   **target_schema**: `edw`\n    *   **target_table**: `allocationrule`\n    *   **target_column**

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, AgentType
from langchain.tools import Tool


def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"


def get_lineage(query: str) -> str:
    """Get lineage for a given query."""

    
    return f"The lineage for the query {query} is {query}"

# Wrap your function as a LangChain Tool
weather_tool = Tool(
    name="get_weather",
    func=get_weather,
    description="Get the weather for a given city."
)

# Choose GPT-4 Omni model
llm = ChatOpenAI(
    model="gpt-4.1-mini",  # or gpt-4.1, gpt-4.1-pro
    temperature=0
)

# Create the Agent
agent = initialize_agent(
    tools=[weather_tool],
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# Run
result = agent.invoke("what is the weather in sf")
print(result)


/var/folders/d3/960_llfs0q11myz6l62g1y140000gn/T/ipykernel_26242/996246303.py:25: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(




> Entering new AgentExecutor chain...
Thought: The question asks for the weather in "sf," which likely refers to San Francisco. I should get the weather for San Francisco.
Action: get_weather
Action Input: San Francisco
Observation: It's always sunny in San Francisco!
Thought:Thought: I now know the final answer
Final Answer: The weather in San Francisco is sunny.

> Finished chain.
{'input': 'what is the weather in sf', 'output': 'The weather in San Francisco is sunny.'}


# Tool 1 Vector QA (column lineage)

In [5]:
from typing import Dict, Any
from langchain.agents import Tool, initialize_agent, AgentType
from app.vector_db import get_qa_chain, LLM

# Configure the org/collection to search
ORG_ID = "76d33fb3-6062-456b-a211-4aec9971f8be"  # <-- change to your org collection name

def get_lineage_answer(question: str) -> str:
    """
    Simple tool function: queries the org's lineage vector store and returns an answer.
    Includes a short source preview for transparency.
    """
    qa = get_qa_chain(org_id=ORG_ID, k=8)
    result: Dict[str, Any] = qa.invoke({"query": question})
    answer = result.get("result", "") or result.get("answer", "")

    # Minimal source preview (first few lines of each Document)
    previews = []
    for doc in result.get("source_documents", []) or []:
        preview = (getattr(doc, "page_content", "") or "").splitlines()[:5]
        previews.append("\n".join(preview))

    sources_block = "\n\nSources:\n- " + "\n- ".join(p for p in previews if p) if previews else ""
    return (answer or "No answer found.") + sources_block

# Wrap as a Tool
extract_lineage_tool = Tool(
    name="extract_lineage",
    func=get_lineage_answer,
    description="Answer lineage questions using the org's vector store. Input is a natural-language question."
)

# Use your existing Gemini LLM from app/vector_db.py
agent = initialize_agent(
    tools=[extract_lineage_tool],
    llm=LLM,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# Example run
resp = agent.invoke("Which are the tables and columns feeds fundlist in dimsharedservicesallocationrule?")
print(resp)



> Entering new AgentExecutor chain...
I need to find the tables and columns that feed into the `fundlist` column of the `dimsharedservicesallocationrule` table. This is a lineage question.
Action: extract_lineage
Action Input: Which tables and columns feed fundlist in dimsharedservicesallocationrule?
Observation: transform_zone.edw.allocationrule.fundlist feeds into transform_zone.edw.dimsharedservicesallocationrule.fundlist.
transform_zone.edw.gl_summary.bookname feeds into transform_zone.edw.allocationrule.fundlist.

Sources:
- source_database: transform_zone
source_schema: edw
source_table: allocationrule
source_column: fundlist
target_database: transform_zone
- source_database: transform_zone
source_schema: edw
source_table: allocationrule
source_column: accountlist
target_database: transform_zone
- source_database: transform_zone
source_schema: edw
source_table: gl_summary
source_column: bookname
target_database: transform_zone
- source_database: transform_zone
source_schema: ed

# Tool 2

In [6]:
from langchain.agents import initialize_agent, AgentType
from app.tools import build_org_lineage_tool, build_org_query_history_tool, LLM

org_id = "76d33fb3-6062-456b-a211-4aec9971f8be"

# Create both tools
lineage_tool = build_org_lineage_tool(org_id=org_id, k=8)
query_history_tool = build_org_query_history_tool(org_id=org_id, max_iters=5)

# Create agent with both tools
agent = initialize_agent(
    tools=[lineage_tool, query_history_tool],
    llm=LLM,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
)

# The agent can now answer both types of questions:
# - Lineage questions: "Which tables feed customer.email?"
# - Impact questions: "I am going to remove column customer.email, what queries are impacted?"
resp = agent.invoke("I am going to remove column gl_summary.deptcd, what queries are impacted?")
print(resp)

2025-11-06 16:55:30,560 [INFO] Loading DATABASE_URL and configuring SQLAlchemy engine
E0000 00:00:1762428330.683670 7628451 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.
E0000 00:00:1762428330.685601 7628451 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.




> Entering new AgentExecutor chain...
I need to find the queries that are impacted by the removal of the column gl_summary.deptcd. I can use the query_history_search tool to find these queries.
Action: query_history_search
Action Input: I will drop column gl_summary.deptcd

/Users/deepkatbamna/Desktop/query_guard/QueryGuard-Backend/app/services/impact_analysis.py:522: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = retriever.get_relevant_documents(q)
2025-11-06 16:55:32,393 [INFO] lineage.retrieval - org=76d33fb3-6062-456b-a211-4aec9971f8be query_len=36 docs=8
2025-11-06 16:55:32,394 [INFO] lineage.retrieval.preview[0]: source_database: transform_zone source_schema: edw source_table: gl_summary source_column: deptcd target_database: transform_zone target_schema: edw target_table: allocationrule target_column: departmentlist query_id: 01c023d5-3202-0e04-0000-0010fc8a5a19 query_type: standalone_insert dependency_scor
2025-11-06 16:55:32,395 [INFO] lineage.retrieval.preview[1]: source_database: transform_zone source_schema: edw source_table: gl_summary source_column: accountcd target_database: transform_zone target_schema: 


Observation: 📊 Impact Analysis Report:
📌 **Change Summary**

Change summary description: Dropping the `deptcd` column from the `gl_summary` table will impact any downstream tables or views that depend on this column. This includes tables that directly ingest data from `gl_summary` or use it in calculations or transformations.

| Field                  | Description |
|------------------------|-------------|
| Change Type            | Drop Column |
| Affected Table         | transform_zone.edw.gl_summary |
| Affected Column(s)     | deptcd |
| Requested Change       | Drop column `deptcd` |
| Reason for Change      | N/A |

---

### **Downstream Impact Analysis**

_List ALL impacted downstream targets grouped by depth._

#### Depth 1 (direct dependencies):
1. **Target Database:** transform_zone
   **Target Schema:** edw
   **Target Table:** allocationrule
   **Target Column:** departmentlist

#### Depth 2:
1. **Target Database:** transform_zone
   **Target Schema:** edw
   **Target Tab

# Tool 3 : PR/Repo Analysis

In [1]:
from app.tools.pr_repo import fetch_pr_analyses_for_org
rows = fetch_pr_analyses_for_org(
    "76d33fb3-6062-456b-a211-4aec9971f8be",
    repo_full_name="Intellytics-Solutions/github-app-poc",
    pr_number=63,
    limit=5,
)
print(len(rows))
print(rows[0] if rows else "no rows")

E0000 00:00:1762434748.820428 7785782 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.
E0000 00:00:1762434748.823934 7785782 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.
2025-11-06 18:42:29,077 [INFO] Loading DATABASE_URL and configuring SQLAlchemy engine
E0000 00:00:1762434749.229835 7785782 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.
E0000 00:00:1762434749.231280 7785782 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


1
{'id': '771dbbbe-15a6-4662-8601-4ca822ca807f', 'org_id': '76d33fb3-6062-456b-a211-4aec9971f8be', 'installation_id': '3afe9067-a516-494a-96fe-6038e4315557', 'repository_id': '33c22b77-9f63-454a-8a40-72713757be8f', 'repo_full_name': 'Intellytics-Solutions/github-app-poc', 'pr_number': 63, 'pr_title': 'Test 1', 'analysis_data': {'files': [{'sql_change': 'File: test_change.sql (added) [+1/-0]\n@@ -0,0 +1 @@\n+ALTER TABLE transform_zone.edw.gl_summary DROP COLUMN deptcd;', 'impact_analysis': '📌 **Change Summary**\n\nChange summary description: _Dropping the `deptcd` column from the `gl_summary` table will impact any downstream tables that depend on this column. This includes tables that use `deptcd` for calculations, filtering, or joining._\n\n| Field                  | Description |\n|------------------------|-------------|\n| Change Type            | Drop Column |\n| Affected Table         | transform_zone.edw.gl_summary |\n| Affected Column(s)     | deptcd |\n| Requested Change       |

In [2]:
from langchain.agents import initialize_agent, AgentType
from app.tools import build_org_pr_repo_tool, LLM

org_id = "76d33fb3-6062-456b-a211-4aec9971f8be"
pr_repo_tool = build_org_pr_repo_tool(org_id=org_id, default_limit=10)

agent = initialize_agent(
    tools=[pr_repo_tool],
    llm=LLM,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
)

resp = agent.invoke("Show PR 63 in Intellytics-Solutions/github-app-poc")
print(resp)

/var/folders/d3/960_llfs0q11myz6l62g1y140000gn/T/ipykernel_32870/1517051530.py:7: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(




> Entering new AgentExecutor chain...
I need to retrieve the analysis for PR 63 in the specified repository.
Action: pr_repo_analysis
Action Input: Intellytics-Solutions/github-app-poc PR 63
Observation: Found 1 stored PR analysis record(s).
1. Intellytics-Solutions/github-app-poc PR #63 - Test 1
   Impact Preview: 📌 **Change Summary**  Change summary description: _Dropping the `deptcd` column from the `gl_summary` table will impact any downstream tables that depend on this column. This includes tables that use `deptcd` for calculations, filtering, or joining._  | Field                  | Description | |------------------------|-------------|

DATA:
{"analyses": [{"id": "771dbbbe-15a6-4662-8601-4ca822ca807f", "org_id": "76d33fb3-6062-456b-a211-4aec9971f8be", "installation_id": "3afe9067-a516-494a-96fe-6038e4315557", "repository_id": "33c22b77-9f63-454a-8a40-72713757be8f", "repo_full_name": "Intellytics-Solutions/github-app-poc", "pr_number": 63, "pr_title": "Test 1", "analysis_data":